# 7. FMR & FNMR Analysis

Evaluate trained Siamese Network on multi-person test dataset

**Metrics:**
- **FMR** (False Match Rate): Impostors are accepted (False Positives)
- **FNMR** (False Non-Match Rate): Genuine are rejected (False Negatives)
- **EER** (Equal Error Rate): Optimal Threshold where FMR ≈ FNMR
- **DET-Curve**: Detection Error Trade-off Visualization

## 1. Import & Setup

In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

# Project
sys.path.insert(0, str(Path.cwd().parent))
import config
from src import data, models, metrics, utils
from src.models import L1Dist

print("✓ Imports successful")

✓ Imports successful


## 2. Load Trained Model

In [2]:
model_path = str(config.CHECKPOINT_DIR / "siamese_final.keras")

print(f"Loading model from: {model_path}\n")

try:
    siamese_model = keras.models.load_model(
        model_path,
        custom_objects={'L1Dist': L1Dist}
    )
    print("✓ model loaded successfully")
except Exception as e:
    print(f"✗ Error loading model: {e}")
    siamese_model = None

Loading model from: C:\Users\angel\Documents\AI Master3.Semester\Doyourecognizeme\checkpoints\siamese_final.keras

✓ model loaded successfully


## 3. Load Test Persons Data

In [3]:
test_persons_dir = Path(config.PROJECT_ROOT) / "data" / "test_persons"

if not test_persons_dir.exists():
    print(f"✗ Test Persons folder not found: {test_persons_dir}")
    print(f"Please collect data first with: python scripts/collect_test_persons.py")
else:
    person_dirs = sorted([d for d in test_persons_dir.iterdir() if d.is_dir()])
    print(f"✓ {len(person_dirs)} Testpersons found:")

    person_images = {}
    for person_dir in person_dirs:
        person_id = person_dir.name
        image_files = list(person_dir.glob("*.jpg"))
        print(f"  {person_id}: {len(image_files)} Images")
        person_images[person_id] = image_files

    print(f"\nTotal: {sum(len(v) for v in person_images.values())} Images")

    # CONDITION: At least 3 test persons required
    if len(person_images) < 3:
        print(f"\n✗ ERROR: You have only {len(person_images)} test person(s), but need at least 3!")
        print(f"\nWhy? To generate impostor pairs, you need different persons.")
        print(f"Collect more test persons with: python scripts/collect_test_persons.py")
        raise ValueError(f"Too few test persons: {len(person_images)} (at least 3 required)")

✓ 3 Testpersons found:
  person_1: 30 Images
  person_2: 30 Images
  person_3: 31 Images

Total: 91 Images


## 4. Generate Pairs

In [ ]:
def generate_pairs(person_images, num_genuine=10, num_impostor=20):
    y_true = []
    pair_info = []

    # Genuine Pairs
    print("Generate Genuine Pairs...")
    genuine_count = 0

    for person_id, image_files in person_images.items():
        if len(image_files) < 2:
            continue

        indices = np.random.choice(len(image_files), min(num_genuine, len(image_files)), replace=False)

        for i in range(0, len(indices)-1, 2):
            if genuine_count >= num_genuine:
                break

            img_path_1 = image_files[indices[i]]
            img_path_2 = image_files[indices[i+1]]

            y_true.append(1)
            pair_info.append(('genuine', person_id, img_path_1, img_path_2))
            genuine_count += 1

    print(f"  ✓ {genuine_count} Genuine Pairs")

    # Impostor Pairs
    print("Generate Impostor Pairs...")
    impostor_count = 0
    person_ids = list(person_images.keys())

    while impostor_count < num_impostor and len(person_ids) >= 2:
        person_1_id, person_2_id = np.random.choice(person_ids, 2, replace=False)

        img_1 = np.random.choice(person_images[person_1_id])
        img_2 = np.random.choice(person_images[person_2_id])

        y_true.append(0)
        pair_info.append(('impostor', person_1_id, img_1, person_2_id, img_2))
        impostor_count += 1

    print(f"  ✓ {impostor_count} Impostor Pairs")

    return np.array(y_true), pair_info

y_true, pair_info = generate_pairs(person_images, num_genuine=10, num_impostor=20)
print(f"\nTotal Pairs: {len(y_true)}")
print(f"  Genuine: {np.sum(y_true == 1)}")
print(f"  Impostor: {np.sum(y_true == 0)}")

## 5. Generate Predictions

In [ ]:
print(f"Generate Predictions for {len(pair_info)} Pairs...\n")

y_pred = []

for idx, pair in enumerate(pair_info):
    try:
        if len(pair) == 4:
            _, person_id, img1_path, img2_path = pair
        else:
            _, person_1_id, img1_path, person_2_id, img2_path = pair

        img1 = data.load_image(str(img1_path))
        img2 = data.load_image(str(img2_path))

        img1_proc = data.preprocess_image(img1).numpy()
        img2_proc = data.preprocess_image(img2).numpy()

        similarity = utils.verify_faces(siamese_model, img1_proc, img2_proc)[0]
        y_pred.append(similarity)

        if (idx + 1) % 10 == 0:
            print(f"  ✓ {idx + 1}/{len(pair_info)} Predictions")

    except Exception as e:
        print(f"  ✗ Error at Pair {idx}: {e}")
        y_pred.append(0.5)

y_pred = np.array(y_pred)
print(f"\n✓ {len(y_pred)} Predictions generated")

## 6. FMR & FNMR Metrics

In [ ]:
thresholds = np.linspace(0.1, 0.99, 20)
results = metrics.calculate_metrics_for_thresholds(y_true, y_pred, thresholds)

eer_value, optimal_threshold = metrics.calculate_eer(y_true, y_pred)

print(f"\n{'='*90}")
print(f"FMR & FNMR METRICS")
print(f"{'='*90}")
print(f"\n{'Threshold':<12} {'FMR':<12} {'FNMR':<12} {'Notes':<40}")
print(f"{'-'*90}")

for result in results:
    threshold = result['threshold']
    fmr = result['fmr']
    fnmr = result['fnmr']

    notes = ""
    if abs(threshold - 0.5) < 0.01:
        notes = "← Standard (current)"
    elif abs(threshold - optimal_threshold) < 0.01:
        notes = "← EER (Optimum)"

    print(f"{threshold:<12.2f} {fmr:<12.4f} {fnmr:<12.4f} {notes:<40}")

print(f"{'-'*90}")
print(f"\nEER: {optimal_threshold:.2f} (Value: {eer_value:.4f})")
print(f"Current: 0.50 (FMR: {metrics.calculate_fmr(y_true, y_pred, 0.5):.4f}, FNMR: {metrics.calculate_fnmr(y_true, y_pred, 0.5):.4f})")
print(f"{'='*90}\n")

## 7. DET-Curve

In [ ]:
fine_thresholds = np.linspace(0.0, 1.0, 50)
fmr_list = [metrics.calculate_fmr(y_true, y_pred, t) for t in fine_thresholds]
fnmr_list = [metrics.calculate_fnmr(y_true, y_pred, t) for t in fine_thresholds]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: FMR vs FNMR
axes[0].plot(fine_thresholds, fmr_list, 'b-', label='FMR', linewidth=2)
axes[0].plot(fine_thresholds, fnmr_list, 'r-', label='FNMR', linewidth=2)
axes[0].axvline(x=0.5, color='green', linestyle='--', linewidth=2, label='Current (0.5)')
axes[0].axvline(x=optimal_threshold, color='orange', linestyle='--', linewidth=2, label=f'EER ({optimal_threshold:.2f})')
axes[0].set_xlabel('Threshold', fontsize=12)
axes[0].set_ylabel('Error Rate', fontsize=12)
axes[0].set_title('FMR & FNMR vs Threshold', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: DET-Curve
axes[1].semilogy(fmr_list, fnmr_list, 'b-', linewidth=2, label='DET Curve')
axes[1].plot(metrics.calculate_fmr(y_true, y_pred, 0.5),
             metrics.calculate_fnmr(y_true, y_pred, 0.5),
             'go', markersize=10, label='Current (0.5)')
axes[1].plot(metrics.calculate_fmr(y_true, y_pred, optimal_threshold),
             metrics.calculate_fnmr(y_true, y_pred, optimal_threshold),
             'r^', markersize=10, label=f'EER ({optimal_threshold:.2f})')
axes[1].set_xlabel('FMR', fontsize=12)
axes[1].set_ylabel('FNMR', fontsize=12)
axes[1].set_title('DET Curve', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Biometric Performance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Score Distribution

In [ ]:
genuine_scores = y_pred[y_true == 1]
impostor_scores = y_pred[y_true == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(genuine_scores, bins=15, alpha=0.7, label=f'Genuine (n={len(genuine_scores)})', color='green', edgecolor='black')
axes[0].hist(impostor_scores, bins=15, alpha=0.7, label=f'Impostor (n={len(impostor_scores)})', color='red', edgecolor='black')
axes[0].axvline(x=0.5, color='blue', linestyle='--', linewidth=2, label='Current (0.5)')
axes[0].axvline(x=optimal_threshold, color='orange', linestyle='--', linewidth=2, label=f'EER ({optimal_threshold:.2f})')
axes[0].set_xlabel('Similarity Score', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Score Distribution', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

data_to_plot = [genuine_scores, impostor_scores]
bp = axes[1].boxplot(data_to_plot, label=['Genuine', 'Impostor'], patch_artist=True)
bp['boxes'][0].set_facecolor('green')
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('red')
bp['boxes'][1].set_alpha(0.7)
axes[1].axhline(y=0.5, color='blue', linestyle='--', linewidth=2, label='Current (0.5)')
axes[1].axhline(y=optimal_threshold, color='orange', linestyle='--', linewidth=2, label=f'EER ({optimal_threshold:.2f})')
axes[1].set_ylabel('Similarity Score', fontsize=12)
axes[1].set_title('Box Plot', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Genuine vs Impostor Score Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Summary

In [ ]:
print(f"\n{'='*80}")
print(f"SUMMARY")
print(f"{'='*80}")

print(f"\nDataset: {len(person_images)} Testpersons")
print(f"Genuine: {np.sum(y_true == 1)}, Impostor: {np.sum(y_true == 0)}")

print(f"\nGenuine Scores: μ={genuine_scores.mean():.4f}, σ={genuine_scores.std():.4f}")
print(f"Impostor Scores: μ={impostor_scores.mean():.4f}, σ={impostor_scores.std():.4f}")

print(f"\nCurrent (0.50): FMR={metrics.calculate_fmr(y_true, y_pred, 0.5):.4f}, FNMR={metrics.calculate_fnmr(y_true, y_pred, 0.5):.4f}")
print(f"Optimum EER ({optimal_threshold:.2f}): FMR={metrics.calculate_fmr(y_true, y_pred, optimal_threshold):.4f}, FNMR={metrics.calculate_fnmr(y_true, y_pred, optimal_threshold):.4f}")

print(f"\n{'='*80}")